### Starter pack for the Brain age prediction EEG data set (Helsinki 1K-Kids) : Data pre-processing

**Last updated:** 31.10.2026

This code reads EDF files (8.7 GB), filters the signal (data cleaning), downsamples to 64 Hz, converts to 16 bit floats to save space and writes as torch files (2.2 GB), In the present version the electrocardiogram (ECG) channel is omitted. The code also generates train/validation/test splits for use in the project. The splits are defined by a list of tuples of the form (0, duration, 0, age, filename) where the filename corresponds to subject ID. The lists are used by the dataloaders for fast loading of the data during training/testing.

The pre-processed files can be downloaded via the Canvas course page.

The original files (not needed if you use the pre-processed files) are here: https://zenodo.org/records/17138539

Requirements: MNE toolbox

In [ ]:
%load_ext autoreload
%autoreload 2

### Configuration

In [ ]:
import os
from pathlib import Path

path = Path('data')
edf_path = path / Path('edf_files')
pre_processed_path = path / Path('pre_processed')

TARGET_DURATION = 901 # Most files (but not all) are 901 seconds

# Downsample from whatever sampling rate used during recording to EEG
# to resampling_rate
RESAMPLING_RATE = 64

# Amplitude cutoff (to avoid overflow/roundoff errors)
MAX_AMPLITUDE = 2000 # microvolt

# Filter settings for EEG pre-processing
LOW_CUT = 0.5
HIGH_CUT = 30.0
NOTCH = 50.0 # The voltage in Europe is 50 Hz

# EEG preprocessing - montage derivation and filter, and feature calculation
montage = ['FP1-F7', 'F7-T3', 'T3-T5', 'T5-O1', 'FP2-F8', 'F8-T4',
           'T4-T6',  'T6-O2', 'T3-C3', 'C3-CZ',  'CZ-C4', # 'A1-T3'
           'C4-T4',  'FP1-F3','F3-C3', 'C3-P3',  'P3-O1', # 'T4-A2'
           'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2']
MONTAGE = {i : chans for i, chans in enumerate(montage)}

### Read the labels

In [ ]:
import pandas as pd

df = pd.read_csv(path / 'anon_codex_age_sex.csv')
print(df.head())
label_mapping = {s[0] : s[1] for s in list(df.itertuples(index=False, name=None))} # s[1] is age, s[2] is gender

### Read EDF files, pre-process and write as Torch files

In [ ]:
import glob
import numpy as np
from edf import Edf, read_edf
from signal_processing import filter_and_resample, clamp 

def load_eeg_file(file_name, file_sampling_rate):
    # Read a single EEG recording from file and store in matrix 'data' (n_channels x n_samples)
    sampling_freq, do2wnsample, dsf, data, channel_labels, n, start_time, _ = read_edf(file_name, file_sampling_rate)

    sampling_freq = int(sampling_freq)
    label2channel={}
    for channel, label in enumerate(channel_labels):
        label2channel[label.upper()] = channel
    return data, sampling_freq, channel_labels, label2channel

def preprocess_eeg(data, label2channel, sampling_rate, RESAMPLING_RATE,
                   low_cut=0.3, high_cut=30, notch=60):
    # Compute montage
    derived = np.empty((len(MONTAGE), data.shape[1]))
    for channel in range(len(MONTAGE)):
        chans = MONTAGE[channel].split('-')
        if chans[0] in label2channel:
            minuend = label2channel[chans[0]]       ######.replace('EEG ', '').replace('-REF','')]
        else:
            minuend = label2channel['EEG ' + chans[0] + '-REF']
        if chans[1] in label2channel:
            subtrahend = label2channel[chans[1]]
        else:
            subtrahend = label2channel['EEG ' + chans[1] + '-REF']
        derived[channel] = data[minuend] - data[subtrahend]
    data_filtered = filter_and_resample(derived, sampling_rate, RESAMPLING_RATE, low_cut, high_cut, notch)
    return data_filtered, montage

segments = []
for i, filename in enumerate(glob.iglob(str(edf_path / '*.edf'), recursive=True)):
    print(i + 1, filename)

    edf = Edf(filename)
    _, start_time, sf, chan, n_samples, _ = edf.return_hdr()

    data, sampling_freq, channel_labels, label2channel = load_eeg_file(filename, None)
    assert(sampling_freq == 250.0)
    data_filtered, montage = preprocess_eeg(data, label2channel,
                                            sampling_freq, RESAMPLING_RATE,
                                            low_cut=LOW_CUT, high_cut=HIGH_CUT, notch=NOTCH)

    # Some recording are slightly shorter than 15 min, zero pad to make all recordings the same length
    npad = int(RESAMPLING_RATE*TARGET_DURATION) - data_filtered.shape[1]
    if npad > 0:
        print("    zero padding ", npad, " samples")
        data_filtered = np.pad(data_filtered, pad_width=((0,0),(0,npad)))
    segments.append((0, TARGET_DURATION, 0, label_mapping[filename[-10:]], filename[-10:]))

    # Pre-processed EEG is stored as float16 values but are converted to float32 values prior to use with Pytorch
    data_filtered = clamp(torch.from_numpy(data_filtered), -MAX_AMPLITUDE, MAX_AMPLITUDE).to(torch.float16)
    torch_file = filename[-10:-4] + '.pt'
    Path(pre_processed_path).mkdir(parents=False, exist_ok=True)
    torch.save(data_filtered, pre_processed_path / torch_file)
print(f'\nNumber of patients {len(segments):d}')

In [ ]:
segments_train, segments_test = train_test_split(segments, test_size=0.2, random_state=42) # 80% goes into train
segments_val, segments_test = train_test_split(segments_test, test_size=0.5, random_state=42) # 10% into val, 10% into test
print("Train:", len(segments_train))
print("Validation:", len(segments_val))
print("Test:", len(segments_test))

torch.save(segments_train, path / 'segments_h1k_train.pt')
torch.save(segments_val, path / 'segments_h1k_val.pt')
torch.save(segments_test, path / 'segments_h1k_test.pt')